In [1]:
import os
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())

config_path = "configs/central-kalimantan.yaml"

from palmdef_risk.io.run import create_or_resume_run
ctx = create_or_resume_run(config_path, resume=True)

Resuming existing run: wri_central-kalimantan_baseline_20260611_173310
run_dir=G:\My Drive\JOB\WRI GDRIVE RS\Deforestation\Deforestation Risk\Technical Parts\deforestation-risk-palmoil v2.0\active\runs\wri_central-kalimantan_baseline_20260611_173310


In [ ]:
import forestatrisk as far

ctx.output_dir.mkdir(parents=True, exist_ok=True)
sample_path = ctx.output_dir / "sample.csv"
if sample_path.exists():
    print(f"sample.csv exists — skipping sampling: {sample_path}")
else:
    far.sample(
        nsamp=ctx.config.nsamp,
        adapt=False,
        seed=ctx.config.seed,
        csize=ctx.config.csize,
        var_dir=str(ctx.data_dir),
        input_forest_raster="fcc23.tif",
        output_file=str(sample_path),
    )

In [3]:
from palmdef_risk.model.diagnostics import compute_vif
covariates = ["altitude", "slope", "dist_defor", "dist_edge", "dist_road",
              "dist_town", "dist_river", "gravity_resid"]
compute_vif(covariates, ctx.output_dir / "sample.csv",
            ctx.output_dir / "diagnostics" / "vif.json")

{'altitude': np.float64(2.15),
 'slope': np.float64(1.764),
 'dist_defor': np.float64(4.207),
 'dist_edge': np.float64(4.128),
 'dist_road': np.float64(2.372),
 'dist_town': np.float64(2.896),
 'dist_river': np.float64(1.131),
 'gravity_resid': np.float64(1.1)}

In [4]:
from palmdef_risk.model.icar import fit_all
fit_all(ctx)

Compute number of 10 x 10 km spatial cells
... 2793 cells (49 x 57)
Identify adjacent cells and compute number of neighbors


Fitting iCAR models:   0%|          | 0/3 [00:00<?, ?variant/s]

Dropped 13 rows with NaN in formula columns


Using estimates from classic logistic regression as starting values for betas


Dropped 13 rows with NaN in formula columns


Using estimates from classic logistic regression as starting values for betas


Dropped 13 rows with NaN in formula columns


Using estimates from classic logistic regression as starting values for betas


[WindowsPath('runs/wri_central-kalimantan_baseline_20260611_173310/output/models/model_A/mod_A.pkl'),
 WindowsPath('runs/wri_central-kalimantan_baseline_20260611_173310/output/models/model_B/mod_B.pkl'),
 WindowsPath('runs/wri_central-kalimantan_baseline_20260611_173310/output/models/model_C/mod_C.pkl')]

In [5]:
from palmdef_risk.model.predict import predict_all
predict_all(ctx)

Predicting risk:   0%|          | 0/3 [00:00<?, ?variant/s]

Write spatial random effect data to disk
Compute statistics
Resampling spatial random effects to file runs\wri_central-kalimantan_baseline_20260611_173310\output\models\model_A\rho.tif
Make virtual raster with variables as raster bands
Divide region in 122 blocks
Create a raster file on disk for projections
Predict deforestation probability by block
forestatrisk: 0...10...20...30...40...50...60...70...80...90...100 - done
Compute statistics
Divide region in 4453 blocks
Compute the number of pixels with value=1
forestatrisk: 0...10...20...30...40...50...60...70...80...90...100 - done
Compute the corresponding area in ha
Divide region in 4453 blocks
Compute the number of pixels with value=1
forestatrisk: 0...10...20...30...40...50...60...70...80...90...100 - done
Compute the corresponding area in ha
Identify threshold
Minimize error on deforested hectares
Create a raster file on disk for forest-cover change
Divide region in 122 blocks
Write raster of future forest-cover change
forestatri

Prediction failed for variant C:
Traceback (most recent call last):
  File "c:\Users\musli\.conda\envs\palmdef-risk\Lib\site-packages\patsy\compat.py", line 40, in call_and_wrap_exc
    return f(*args, **kwargs)
  File "c:\Users\musli\.conda\envs\palmdef-risk\Lib\site-packages\patsy\eval.py", line 179, in eval
    return eval(code, {}, VarLookupDict([inner_namespace] + self._namespaces))
  File "<string>", line 1, in <module>
  File "c:\Users\musli\.conda\envs\palmdef-risk\Lib\site-packages\patsy\mgcv_cubic_splines.py", line 709, in transform
    dm = _get_crs_dmatrix(
        x, self._all_knots, self._constraints, cyclic=self._cyclic
    )
  File "c:\Users\musli\.conda\envs\palmdef-risk\Lib\site-packages\patsy\mgcv_cubic_splines.py", line 372, in _get_crs_dmatrix
    dm = _get_free_crs_dmatrix(x, knots, cyclic)
  File "c:\Users\musli\.conda\envs\palmdef-risk\Lib\site-packages\patsy\mgcv_cubic_splines.py", line 349, in _get_free_crs_dmatrix
    dmt = ajm * i[j, :].T + ajp * i[j1, :].T 

[WindowsPath('runs/wri_central-kalimantan_baseline_20260611_173310/output/predictions/risk_A.tif'),
 WindowsPath('runs/wri_central-kalimantan_baseline_20260611_173310/output/predictions/forest_future_A.tif'),
 WindowsPath('runs/wri_central-kalimantan_baseline_20260611_173310/output/predictions/risk_B.tif'),
 WindowsPath('runs/wri_central-kalimantan_baseline_20260611_173310/output/predictions/forest_future_B.tif')]

In [6]:
from palmdef_risk.model.diagnostics import compute_residuals_all, compute_morans_i
residuals, coords = compute_residuals_all(ctx)
compute_morans_i(residuals, coords, ctx.output_dir / "diagnostics" / "moran.json")

Computing deviance residuals:   0%|          | 0/3 [00:00<?, ?variant/s]

{'A': {'I': 0.2539, 'p_value': 0.0, 'n': 199987},
 'B': {'I': 0.2538, 'p_value': 0.0, 'n': 199987},
 'C': {'I': 0.2532, 'p_value': 0.0, 'n': 199987}}

In [7]:
from palmdef_risk.model.sensitivity import run_gravity_sensitivity
run_gravity_sensitivity(ctx)

Gravity sensitivity (σ sweep):   0%|          | 0/3 [00:00<?, ?σ/s]

Dropped 13 rows with NaN in formula columns


Compute number of 10 x 10 km spatial cells
... 2793 cells (49 x 57)
Identify adjacent cells and compute number of neighbors
Using estimates from classic logistic regression as starting values for betas


Dropped 13 rows with NaN in formula columns


Compute number of 10 x 10 km spatial cells
... 2793 cells (49 x 57)
Identify adjacent cells and compute number of neighbors
Using estimates from classic logistic regression as starting values for betas


Dropped 13 rows with NaN in formula columns


Compute number of 10 x 10 km spatial cells
... 2793 cells (49 x 57)
Identify adjacent cells and compute number of neighbors
Using estimates from classic logistic regression as starting values for betas


WindowsPath('runs/wri_central-kalimantan_baseline_20260611_173310/output/diagnostics/gravity_sensitivity.json')

In [2]:
from palmdef_risk.model.reports import export_all_diagnostics

results = export_all_diagnostics(ctx)
n = 0
for group, paths in results.items():
    label = "run-level" if group == "_run" else f"model {group}"
    for p in paths:
        n += 1
        print(f"output {n} [{label}]: {p}")

Run-level diagnostics:   0%|          | 0/2 [00:00<?, ?artifact/s]

Per-variant diagnostics:   0%|          | 0/18 [00:00<?, ?artifact/s]

output 1 [run-level]: runs\wri_central-kalimantan_baseline_20260611_173310\output\diagnostics\csize_icar.txt
output 2 [run-level]: runs\wri_central-kalimantan_baseline_20260611_173310\output\diagnostics\fcc_history.png
output 3 [model A]: runs\wri_central-kalimantan_baseline_20260611_173310\output\diagnostics\A\summary_icar.txt
output 4 [model A]: runs\wri_central-kalimantan_baseline_20260611_173310\output\diagnostics\A\roc_calibration.png
output 5 [model A]: runs\wri_central-kalimantan_baseline_20260611_173310\output\diagnostics\A\accuracy_summary.txt
output 6 [model A]: runs\wri_central-kalimantan_baseline_20260611_173310\output\diagnostics\A\mcmc_traces.png
output 7 [model A]: runs\wri_central-kalimantan_baseline_20260611_173310\output\diagnostics\A\mcmc_autocorr.png
output 8 [model A]: runs\wri_central-kalimantan_baseline_20260611_173310\output\diagnostics\A\mcmc_ess.txt
output 9 [model A]: runs\wri_central-kalimantan_baseline_20260611_173310\output\diagnostics\A\risk_map.png
outpu